# E1 — YOLOv8-nano Baseline Detection

**Owner:** Aly  
**Track:** Detection  
**Model:** YOLOv8n (nano)  

| | |
|---|---|
| **Input** | `dataset_split_70_15_15/data.yaml` (70/15/15 stratified split) |
| **Weights output** | `models/yolo/yolo_baseline_E1.pt` |
| **Metrics output** | `results/metrics/detection_results.csv` |
| **Predictions** | `results/predictions/E1_yolo_predictions.csv` |
| **Figures** | `figures/yolo/` |
| **Log** | `results/logs/training_logs/E1_training_results.csv` |

**Dependencies:** None — runs independently after the dataset is in Google Drive.  
**Estimated training time on T4:** ~2–4 h for 100 epochs (early stopping at patience=20).  

---
**Metrics reported:** mAP@0.5, mAP@0.5:0.95, Precision, Recall, FPS, Model size (MB), per-class mAP

In [1]:
# ── USER CONFIGURATION ────────────────────────────────────────────────────────
# Auto-detects repo root from notebook location (works in VS Code / Jupyter).
from pathlib import Path

_here = Path().resolve()
REPO_ROOT = _here.parent if _here.name == "notebooks" else _here

print(f"REPO_ROOT : {REPO_ROOT}")
# Uncomment & set manually if auto-detect is wrong:
# REPO_ROOT = Path(r"C:\Users\zaki\Desktop\aly eui\Computer_Vision_Latest")
# ─────────────────────────────────────────────────────────────────────────────

REPO_ROOT : /content


In [2]:
# Install once if not already installed:
#   pip install ultralytics>=8.0.0
import importlib, subprocess, sys
if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics>=8.0.0", "-q"])
print("ultralytics ready")

ultralytics ready


In [3]:
import sys
sys.path.insert(0, str(REPO_ROOT))
print(f"sys.path updated: {REPO_ROOT}")

sys.path updated: /content


In [4]:
import os
import shutil
import time
import yaml
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.seed import set_seed, SEED
from src.utils.paths import CLASSES, CLASS_TO_IDX
from src.utils.config_loader import load_config
from src.utils.logger import get_logger
from src.evaluation.detection_metrics import parse_yolo_results, get_model_size_mb

set_seed()
cfg = load_config(f"{REPO_ROOT}/config.yaml")
logger = get_logger("E1_YOLO")

# ── Output paths inside the repo ─────────────────────────────────────────────
REPO             = Path(REPO_ROOT)
MODELS_YOLO      = REPO / "models" / "yolo"
CKPT_LAST        = REPO / "models" / "checkpoints" / "last"
RESULTS_METRICS  = REPO / "results" / "metrics"
RESULTS_PREDS    = REPO / "results" / "predictions"
FIGURES_YOLO     = REPO / "figures" / "yolo"
LOGS_TRAINING    = REPO / "results" / "logs" / "training_logs"

for d in [MODELS_YOLO, CKPT_LAST, RESULTS_METRICS, RESULTS_PREDS, FIGURES_YOLO, LOGS_TRAINING]:
    d.mkdir(parents=True, exist_ok=True)

# ── Colab-local training workspace (fast I/O, not Drive) ─────────────────────
COLAB_TRAIN = REPO_ROOT / "data" / "processed" / "yolo_training"
SPLIT_OUT   = REPO_ROOT / "data" / "processed" / "dataset_split_70_15_15"
DATA_YAML   = SPLIT_OUT / "data.yaml"

logger.info("Environment ready")

ModuleNotFoundError: No module named 'src'

In [ ]:
# Smoke-test all src imports before training starts
from src.data.split_dataset import build_image_inventory, make_split, get_dominant_class
from src.evaluation.detection_metrics import parse_yolo_results, get_model_size_mb
from src.evaluation.plots import plot_confusion_matrix
print("All src imports OK")

## Step 1 — Dataset Preparation

Verifies the 70/15/15 stratified split exists (produced by `01_preprocessing_and_splits.ipynb`).

> Run `01_preprocessing_and_splits.ipynb` first — it creates `data/processed/dataset_split_70_15_15/`.

In [ ]:
# The split is produced by notebooks/01_preprocessing_and_splits.ipynb.
# Run that notebook first, then come back here.
if DATA_YAML.exists():
    logger.info(f"Split found at {SPLIT_OUT} — ready to train")
    print(f"Split found: {SPLIT_OUT}")
    print(f"data.yaml  : {DATA_YAML}")
else:
    raise FileNotFoundError(
        f"data.yaml not found at {DATA_YAML}\n"
        "Run notebooks/01_preprocessing_and_splits.ipynb first."
    )

In [ ]:
# Verify split integrity
print(f"{'Split':<8} {'Images':>8} {'Labels':>8}")
print("-" * 26)
for split in ["train", "val", "test"]:
    n_img = len(list((COLAB_SPLIT / split / "images").glob("*.jpg")))
    n_lbl = len(list((COLAB_SPLIT / split / "labels").glob("*.txt")))
    match = "OK" if n_img == n_lbl else "MISMATCH"
    print(f"{split:<8} {n_img:>8} {n_lbl:>8}   {match}")

assert DATA_YAML.exists(), "data.yaml not found — re-run the preparation cell"
print(f"\ndata.yaml: {DATA_YAML}")
print("Dataset ready")

## Step 2 — Train YOLOv8n

All hyperparameters are taken from `configs/yolo_config.yaml`. Locked constants (seed, imgsz, conf, iou) from `config.yaml`.

Training writes to `/content/yolo_e1/` (fast Colab local storage). Weights are copied to the repo after training completes.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

train_results = model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=416,
    batch=16,
    patience=20,
    project=str(COLAB_TRAIN),
    name="E1_baseline",
    seed=SEED,
    exist_ok=True,
    verbose=True,
    # Augmentation (from configs/yolo_config.yaml)
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=0.0, translate=0.1, scale=0.5,
    flipud=0.0, fliplr=0.5,
    mosaic=1.0, mixup=0.0,
    conf=0.25, iou=0.45,
)

TRAIN_RUN_DIR = COLAB_TRAIN / "E1_baseline"
print(f"\nTraining complete: {TRAIN_RUN_DIR}")

## Step 3 — Save Weights and Training Log to Repo

In [ ]:
BEST_PT = TRAIN_RUN_DIR / "weights" / "best.pt"
LAST_PT = TRAIN_RUN_DIR / "weights" / "last.pt"

REPO_BEST = MODELS_YOLO / "yolo_baseline_E1.pt"
REPO_LAST = CKPT_LAST   / "yolo_E1_last.pt"

shutil.copy2(BEST_PT, REPO_BEST)
shutil.copy2(LAST_PT, REPO_LAST)

# Training loss/metric curves CSV
shutil.copy2(TRAIN_RUN_DIR / "results.csv", LOGS_TRAINING / "E1_training_results.csv")

print(f"Best weights  -> {REPO_BEST}")
print(f"Last weights  -> {REPO_LAST}")
print(f"Training log  -> {LOGS_TRAINING / 'E1_training_results.csv'}")

## Step 4 — Evaluate on Test Set

Uses `best.pt` on the held-out test split (not seen during training or early stopping).

In [ ]:
best_model = YOLO(str(REPO_BEST))

test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=416,
    conf=0.25,
    iou=0.45,
    verbose=True,
)

rd = test_metrics.results_dict
print("\nTest metrics:")
for k, v in rd.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# FPS benchmark — single-image inference on 100 test images
test_images = sorted((SPLIT_OUT / "test" / "images").glob("*.jpg"))

# Two warm-up passes
for _ in range(2):
    best_model.predict(str(test_images[0]), conf=0.25, iou=0.45, verbose=False)

BENCH_N = min(100, len(test_images))
times_ms = []
for img_path in test_images[:BENCH_N]:
    t0 = time.perf_counter()
    best_model.predict(str(img_path), conf=0.25, iou=0.45, verbose=False)
    times_ms.append((time.perf_counter() - t0) * 1000)

inference_ms = float(np.mean(times_ms))
fps          = 1000.0 / inference_ms
model_mb     = get_model_size_mb(str(REPO_BEST))

print(f"Inference (mean over {BENCH_N} images): {inference_ms:.1f} ms")
print(f"FPS : {fps:.1f}")
print(f"Model size: {model_mb:.2f} MB")

## Step 5 — Save All Results to Repo

In [ ]:
# ── Per-class mAP ────────────────────────────────────────────────────────────
# ap_class_index maps result array positions to class IDs (handles missing classes)
class_indices   = list(test_metrics.box.ap_class_index)      # e.g. [0,1,2,3,4,5]
per_class_ap50  = list(test_metrics.box.ap50)                 # mAP@0.5 per class
per_class_ap5095 = list(test_metrics.box.maps)                # mAP@0.5:0.95 per class

# Build full 6-element arrays (NaN for any missing class)
ap50_by_class   = {CLASSES[i]: float("nan") for i in range(6)}
ap5095_by_class = {CLASSES[i]: float("nan") for i in range(6)}
for idx, ap50_val, ap5095_val in zip(class_indices, per_class_ap50, per_class_ap5095):
    ap50_by_class[CLASSES[idx]]   = round(float(ap50_val), 4)
    ap5095_by_class[CLASSES[idx]] = round(float(ap5095_val), 4)

training_log = pd.read_csv(LOGS_TRAINING / "E1_training_results.csv")

# ── Build detection_results row ──────────────────────────────────────────────
detection_row = {
    "experiment"    : "E1",
    "model"         : "yolov8n",
    "timestamp"     : datetime.now().strftime("%Y-%m-%d %H:%M"),
    "mAP50"         : round(float(rd.get("metrics/mAP50(B)",    float("nan"))), 4),
    "mAP50_95"      : round(float(rd.get("metrics/mAP50-95(B)", float("nan"))), 4),
    "precision"     : round(float(rd.get("metrics/precision(B)",float("nan"))), 4),
    "recall"        : round(float(rd.get("metrics/recall(B)",   float("nan"))), 4),
    "fps"           : round(fps, 2),
    "inference_ms"  : round(inference_ms, 2),
    "model_size_mb" : round(model_mb, 3),
    "epochs_trained": len(training_log),
    "image_size"    : 416,
    "dataset_split" : "70/15/15",
    # Per-class mAP@0.5
    **{f"mAP50_{cls}": ap50_by_class[cls] for cls in CLASSES},
    # Per-class mAP@0.5:0.95
    **{f"mAP5095_{cls}": ap5095_by_class[cls] for cls in CLASSES},
}

# Append/replace E1 row in detection_results.csv
csv_path = RESULTS_METRICS / "detection_results.csv"
if csv_path.exists():
    existing = pd.read_csv(csv_path)
    existing = existing[existing["experiment"] != "E1"]
    df_out = pd.concat([existing, pd.DataFrame([detection_row])], ignore_index=True)
else:
    df_out = pd.DataFrame([detection_row])

df_out.to_csv(csv_path, index=False)
print(f"Saved detection metrics -> {csv_path}")

# Human-readable summary columns
summary_cols = ["experiment", "mAP50", "mAP50_95", "precision", "recall",
                "fps", "inference_ms", "model_size_mb", "epochs_trained"]
print(df_out[summary_cols].to_string(index=False))

In [ ]:
# Per-class metrics detail file (for notebook 09 analysis)
per_class_rows = []
for cls in CLASSES:
    per_class_rows.append({
        "experiment": "E1",
        "class":      cls,
        "mAP50":      ap50_by_class[cls],
        "mAP50_95":   ap5095_by_class[cls],
    })

per_class_path = RESULTS_METRICS / "E1_per_class_metrics.csv"
pd.DataFrame(per_class_rows).to_csv(per_class_path, index=False)
print(f"Per-class metrics -> {per_class_path}")
print(pd.DataFrame(per_class_rows).to_string(index=False))

In [ ]:
# Run inference on all test images and save predictions CSV
# Uses stream=True to avoid loading all results into memory at once
pred_records = []

gen = best_model.predict(
    [str(p) for p in test_images],
    conf=0.25,
    iou=0.45,
    verbose=False,
    stream=True,
)

for result in gen:
    img_name = Path(result.path).name
    for box in result.boxes:
        cls_id = int(box.cls.item())
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        pred_records.append({
            "image"      : img_name,
            "class_id"   : cls_id,
            "class_name" : CLASSES[cls_id],
            "confidence" : round(float(box.conf.item()), 4),
            "x1": round(x1, 1), "y1": round(y1, 1),
            "x2": round(x2, 1), "y2": round(y2, 1),
        })

preds_path = RESULTS_PREDS / "E1_yolo_predictions.csv"
pd.DataFrame(pred_records).to_csv(preds_path, index=False)
print(f"Saved {len(pred_records)} predictions -> {preds_path}")

In [ ]:
# Copy all figures from the Ultralytics training run to figures/yolo/
copied = []
for pattern in ["*.png", "val_batch*.jpg"]:
    for src in TRAIN_RUN_DIR.glob(pattern):
        shutil.copy2(src, FIGURES_YOLO / src.name)
        copied.append(src.name)

# Also copy the confusion matrix from val run if it landed somewhere else
for src in sorted(TRAIN_RUN_DIR.rglob("confusion_matrix*.png")):
    dst = FIGURES_YOLO / src.name
    if not dst.exists():
        shutil.copy2(src, dst)
        copied.append(src.name)

print(f"Copied {len(copied)} figures to {FIGURES_YOLO}:")
for f in sorted(copied):
    print(f"  {f}")

In [ ]:
# ── Final Results Summary ─────────────────────────────────────────────────────
print("=" * 58)
print("E1  YOLOv8-nano Baseline  |  TEST SET RESULTS")
print("=" * 58)
print(f"  mAP@0.5          : {detection_row['mAP50']:.4f}")
print(f"  mAP@0.5:0.95     : {detection_row['mAP50_95']:.4f}")
print(f"  Precision        : {detection_row['precision']:.4f}")
print(f"  Recall           : {detection_row['recall']:.4f}")
print(f"  FPS              : {detection_row['fps']:.1f}")
print(f"  Inference/image  : {detection_row['inference_ms']:.1f} ms")
print(f"  Model size       : {detection_row['model_size_mb']:.2f} MB")
print(f"  Epochs trained   : {detection_row['epochs_trained']}")
print()
print("Per-class mAP@0.5:")
for cls in CLASSES:
    val = ap50_by_class[cls]
    bar = "#" * int(val * 40) if not (val != val) else "(nan)"
    print(f"  {cls:<16} {val:.4f}  {bar}")
print("=" * 58)
print()
print("Files saved:")
print(f"  models/yolo/yolo_baseline_E1.pt")
print(f"  results/metrics/detection_results.csv")
print(f"  results/metrics/E1_per_class_metrics.csv")
print(f"  results/predictions/E1_yolo_predictions.csv")
print(f"  results/logs/training_logs/E1_training_results.csv")
print(f"  figures/yolo/  ({len(copied)} files)")